In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch

#查看训练器版本

In [ ]:
import transformers
print(transformers.__version__)

# 加载预训练模型和分词器
如果速度慢，修改指向
export HF_ENDPOINT=https://hf-mirror.com

In [ ]:
model_name = "defog/llama-3-sqlcoder-8b"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto", #启用了DeepSpeed 不再需要
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 在初始化完model和tokenizer后 设置 pad_token

将 eos_token 作为填充标记，确保训练过程中可以正确处理填充。

In [ ]:
# 设置填充标记为 eos_token
tokenizer.pad_token = tokenizer.eos_token

# 如果调整了特殊标记，更新模型的词汇表大小
model.resize_token_embeddings(len(tokenizer))

# 加载微调数据集

先安装dataset 
pip install datasets

In [ ]:
from datasets import load_dataset
#load_dataset 默认会尝试从 Hugging Face Hub 上加载数据集
#dataset = load_dataset("train_test.json")

#本地加载数据集
# 指定本地文件路径
dataset = load_dataset("json", data_files="train_test.json")

In [ ]:
train_model_name = "../train_model/kpaas_train_model"

#清理GPU缓存

In [ ]:
# Step 2: 清理 GPU 缓存
# torch.cuda.empty_cache()

# 使用 DeepSpeed 的 ZeRO-Offload
DeepSpeed 支持将参数、梯度或优化器状态溢出到 CPU 或磁盘，从而缓解显存不足问题。

In [ ]:
# 定义 DeepSpeed 配置文件路径
ds_config = "deepspeed_config.json"

# 定义训练参数

安装deepspeed
pip install deepspeed

如果安装失败，则确保确保 setuptools 和 pip 是最新版本：
pip install --upgrade pip setuptools packaging

然后再重新安装
pip install deepspeed


In [ ]:
training_args = TrainingArguments(
    output_dir=train_model_name,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=5000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=1000,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True  # 如果有 GPU，开启混合精度训练
    #,deepspeed=ds_config,  # DeepSpeed 配置
    #local_rank=-1  # 禁用分布式训练,启用单机训练
)

# 检查模型是否已被部分加载到 CPU 或磁盘


使用 accelerate 加载模型时，确保模型未被部分卸载。如果您需要显式加载到 GPU，可以在创建 Trainer 之前执行以下操作：

In [ ]:
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from accelerate import Accelerator
import os



# TrainingArguments 中配置了 deepspeed，Trainer 会自动使用 Accelerator 初始化并将其与 DeepSpeed 配合使用。
#如果启用deepspeed，则这里不用显示初始化accelerator

# 初始化 Accelerator
accelerator = Accelerator()

# 初始化模型
with init_empty_weights():
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

#定义权重文件夹路径（model-00001-of-00004.safetensors这些就是项目的权重文件）
path_to_checkpoint=os.path.expanduser("~/.cache/huggingface/hub/models--defog--llama-3-sqlcoder-8b/snapshots/0f96d32e16737bda1bbe0d8fb13a932a8a3fa0bb/")

#定义卸载模块路径，模型的子模块会被卸载到该文件夹，节省一部分内存。
offload_folder="/mnt/workspace/sqlcoder/train/offload/"
    
# 从保存点加载并分发
model = load_checkpoint_and_dispatch(
    model, 
    path_to_checkpoint, 
    device_map="auto", #如果启用了DeepSpeed 不再需要
    offload_folder=offload_folder  # 添加 offload_folder 参数
)



# （停用！停用）创建 Trainer
原因： 上面load_checkpoint_and_dispatch使用了offload_folder解决内存问题，使得部分模块被卸载到磁盘，会导致Trainer 无法正确处理。

<!-- # 使用 Accelerator 调度模型
# model = accelerator.prepare(model)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     tokenizer=tokenizer,
# ) -->

In [ ]:
print(dataset["train"][0])  # 替换 "train" 为你的数据集分区名称

# 创建Trainer前先做数据的预处理

In [ ]:
# 数据预处理
# def preprocess_function(examples):
#     encoded = tokenizer(examples["text"], padding="max_length", truncation=True)
#     encoded["labels"] = encoded["input_ids"].copy()  # 添加 labels
#     return encoded

def preprocess_function(examples):
    # 将批量的字段拼接起来
    texts = [
        instruction + " " + context + " " + input_text + " " + output_text
        for instruction, context, input_text, output_text in zip(
            examples["instruction"], examples["context"], examples["input"], examples["output"]
        )
    ]
    # 对每条拼接的文本进行编码
    encoded = tokenizer(texts, padding="max_length", truncation=True)
    # 添加 labels
    encoded["labels"] = encoded["input_ids"]
    return encoded

encoded_dataset = dataset.map(preprocess_function, batched=True)

# 检查数据集结构
print(encoded_dataset["train"][0])


# 创建自定义Trainer

In [ ]:
# 自定义 Trainer 以跳过设备迁移逻辑
class CustomTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        # 重写此方法，避免 Trainer 将模型强制迁移到 GPU 或其他设备
        pass
    


In [ ]:
# 初始化 Trainer
# trainer = CustomTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     tokenizer=tokenizer,
# )


# 初始化 预处理后的数据进行Trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    tokenizer=tokenizer,
)

# 开始微调

In [ ]:
trainer.train()

# 保存微调后的模型

In [ ]:
model.save_pretrained(train_model_name)
tokenizer.save_pretrained(train_model_name)

In [ ]:
# !export PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:128